In [ ]:
## Import Required Libraries

In [1]:
# TensorFlow library for deep learning
import tensorflow as tf

# Numerical operations
import numpy as np

# Data analysis
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# File and folder operations
import os

# Image preprocessing utility
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Import pretrained DenseNet169 model
from tensorflow.keras.applications import DenseNet169

# Import layers and model builder
from tensorflow.keras import layers, models

# Evaluation metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
# Verify TensorFlow Installation

In [ ]:
# Display installed TensorFlow version
print("TensorFlow Version:", tf.__version__)

In [ ]:
# Define Dataset Path

In [ ]:
# Path to Pap smear dataset
dataset_path = "../dataset"

# Display class folders
print(os.listdir(dataset_path))

In [ ]:
# Count Images Per Class

In [ ]:
# Purpose:

# verify dataset integrity
# identify class imbalance
# confirm images loaded correctly

# Loop through dataset folders
for folder in os.listdir(dataset_path):

    # Create full path
    path = os.path.join(dataset_path, folder)

    # Check whether it is a valid directory
    if os.path.isdir(path):

        # Display folder name and number of images
        print(folder, len(os.listdir(path)))

In [ ]:
# Load and Preprocess Images

In [ ]:
# Purpose:

# resize images
# normalize images
# split training and validation data automatically

# ==============
# Define image size
IMG_SIZE = 224

# Number of images processed at once
BATCH_SIZE = 32

# Image generator for preprocessing
datagen = ImageDataGenerator(

    # Normalize pixel values
    rescale=1./255,

    # Split validation data
    validation_split=0.2
)

# Load training images
train_data = datagen.flow_from_directory(

    # Dataset location
    dataset_path,

    # Resize all images
    target_size=(IMG_SIZE, IMG_SIZE),

    # Batch size
    batch_size=BATCH_SIZE,

    # Multi-class classification
    class_mode='categorical',

    # Training subset
    subset='training'
)

# Load validation images
val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
# Display Class Labels

In [ ]:
# Print class labels and assigned indices
print(train_data.class_indices)

In [ ]:
# Visualize Sample Images
# Purpose:

#verify image loading
#inspect image quality
#detect corrupted images

In [ ]:
# Retrieve one batch of images
images, labels = next(train_data)

# Create plotting figure
plt.figure(figsize=(12,12))

# Display first 9 images
for i in range(9):

    # Create subplot
    plt.subplot(3,3,i+1)

    # Display image
    plt.imshow(images[i])

    # Remove axis
    plt.axis('off')

# Display images
plt.show()

In [ ]:
# Build DenseNet169 Model 
# Purpose:

# use transfer learning
# reuse pretrained image features
# reduce training time
# improve performance

In [ ]:
# Load pretrained DenseNet169 model
base_model = DenseNet169(

    # Use pretrained ImageNet weights
    weights='imagenet',

    # Remove original classifier
    include_top=False,

    # Input image dimensions
    input_shape=(224,224,3)
)

# Freeze pretrained layers
base_model.trainable = False

# Build custom classification model
model = models.Sequential([

    # DenseNet feature extractor
    base_model,

    # Convert feature maps to vectors
    layers.GlobalAveragePooling2D(),

    # Fully connected layer
    layers.Dense(128, activation='relu'),

    # Reduce overfitting
    layers.Dropout(0.5),

    # Final classification layer
    layers.Dense(5, activation='softmax')
])

In [ ]:
# Compile Model

In [ ]:
# Configure training settings
model.compile(

    # Optimization algorithm
    optimizer='adam',

    # Multi-class classification loss function
    loss='categorical_crossentropy',

    # Evaluation metric
    metrics=['accuracy']
)

In [4]:
# View Model Architecture
# Purpose:

#inspect layers
#verify parameters
#confirm architecture correctness

In [ ]:
# Display model architecture summary
model.summary()

In [ ]:
# Train DenseNet169 Model
# Purpose:

#learn image patterns
#classify Pap smear cells
#optimize CNN weights

In [ ]:
# Train the model
history = model.fit(

    # Training dataset
    train_data,

    # Validation dataset
    validation_data=val_data,

    # Number of epochs
    epochs=10
)

In [ ]:
# Plot Training Accuracy
# Purpose:

#monitor learning
#identify overfitting
#compare training vs validation performance

In [ ]:
# Plot training accuracy
plt.plot(
    history.history['accuracy'],
    label='Training Accuracy'
)

# Plot validation accuracy
plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

# Axis labels
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

# Display legend
plt.legend()

# Show graph
plt.show()

In [ ]:
# Plot Training Loss
#Purpose:

#analyze optimization
#monitor convergence
#identify instability

In [ ]:
# Plot training loss
plt.plot(
    history.history['loss'],
    label='Training Loss'
)

# Plot validation loss
plt.plot(
    history.history['val_loss'],
    label='Validation Loss'
)

# Axis labels
plt.xlabel('Epoch')
plt.ylabel('Loss')

# Display legend
plt.legend()

# Show graph
plt.show()

In [ ]:
# Evaluate Model Performance

In [ ]:
# Evaluate model using validation dataset
loss, accuracy = model.evaluate(val_data)

# Display validation accuracy
print("Validation Accuracy:", accuracy)

In [ ]:
# Generate Predictions
#Purpose:

#compare predictions vs actual labels
#prepare evaluation metrics

In [ ]:
# Predict image classes
predictions = model.predict(val_data)

# Convert probabilities into class labels
predicted_classes = np.argmax(predictions, axis=1)

# Retrieve actual class labels
true_classes = val_data.classes

# Retrieve class names
class_labels = list(val_data.class_indices.keys())

In [ ]:
# Classification Report
# Purpose:

#evaluate class-wise performance
#measure precision and recall
#assess model quality

In [ ]:
# Display precision, recall and F1-score
print(classification_report(
    true_classes,
    predicted_classes,
    target_names=class_labels
))

In [ ]:
# Generate Confusion Matrix
# Purpose:

# visualize classification errors
# identify confusing classes
# analyze class performance

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(
    true_classes,
    predicted_classes
)

# Create figure
plt.figure(figsize=(8,6))

# Display heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels
)

# Axis labels
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Title
plt.title('Confusion Matrix')

# Display heatmap
plt.show()

In [ ]:
# Save Trained Model
# Purpose:

#reuse trained model later
#evaluation
#Grad-CAM visualization
#deployment
#future experiment

In [ ]:
# Save trained DenseNet169 model
model.save("../models/densenet169_papsmear.h5")